## Ngày 3 về RAG

### Chuyên gia hỏi đáp cho InsureLLM

Triển khai một pipeline RAG bằng LangChain 1.0.

Sử dụng VectorStore mà chúng ta đã tạo lần trước (với Hugging Face `all-MiniLM-L6-v2`)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [ ]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

### Kết nối với Chroma; sử dụng Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Thiết lập 2 đối tượng LangChain chính: retriever và llm

#### Giải thích thêm về "temperature":
- Kiểm soát mức độ đa dạng của đầu ra
- Temperature bằng 0 nghĩa là đầu ra sẽ có tính dự đoán được
- Temperature cao hơn giúp câu trả lời đa dạng hơn

Một số người mô tả temperature giống như "độ sáng tạo", nhưng cách hiểu đó chưa hoàn toàn chính xác.
- Thực chất, nó kiểm soát việc lựa chọn token trong quá trình suy luận
- temperature=0 nghĩa là: luôn chọn token có xác suất cao nhất
- temperature=1 thường nghĩa là: một token có xác suất 10% sẽ được chọn trong 10% số lần

Lưu ý: temperature bằng 0 không có nghĩa là đầu ra luôn có thể tái lập. Bạn cũng cần đặt một hạt giống ngẫu nhiên. Chúng ta sẽ thực hiện việc đó trong các tuần 6–8. (Ngay cả khi đó, kết quả cũng không phải lúc nào cũng tái lập được.)

Lưu ý 2: nếu muốn có tính sáng tạo, hãy sử dụng System Prompt!

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### Các đối tượng LangChain này triển khai phương thức `invoke()`

In [ ]:
retriever.invoke("Avery là ai?")

In [ ]:
llm.invoke("Avery là ai?")

## Đã đến lúc ghép mọi thứ lại với nhau!

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
Bạn là một trợ lý am hiểu và thân thiện, đại diện cho công ty Insurellm.
Bạn đang trò chuyện với người dùng về Insurellm.
Nếu phù hợp, hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi.
Nếu không biết câu trả lời, hãy nói rõ điều đó.
Ngữ cảnh:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Averi Lancaster là ai?", [])

## Tiếp theo có thể là gì nhỉ? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Thừa nhận đi — bạn đã nghĩ RAG sẽ phức tạp hơn thế này nhiều đúng không!!